<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day10_practice2_real_detect_%EC%99%84%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#_real_detext 완성 - 서버 함수를 만든다

In [2]:
!pip install -U ultralytics

In [3]:
from ultralytics import YOLO
import urllib.request, os, json

In [4]:
BUS_URL = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg"
if not os.path.exists("bus.jpg"):
  urllib.request.urlretrieve(BUS_URL, "bus.jpg")
  print("bus.jpg 다운로드 완료 (재실행 시 재사용)")

In [5]:
# 셀 1. 출력 계약 확인 - mock이 웹 반환하는지가 기준
# 서버 mock 이 반환하는 감지 1건의 형식

출력_예시 = { # 딕셔너리 ("키":값,)
    "type": "포트홀",
    "confidence": 0.92,
    "severity": 4,
    "box": {"x1":0.31,"y1":0.42,"x2":0.58,"y2":0.71},
}

print("지켜야 할 출력 형태:", json.dumps(출력_예시, ensure_ascii=False)) # 딕셔너리 -> JSON 문자열, ensure_ascii=False로 한글 안 깨짐

지켜야 할 출력 형태: {"type": "포트홀", "confidence": 0.92, "severity": 4, "box": {"x1": 0.31, "y1": 0.42, "x2": 0.58, "y2": 0.71}}


In [12]:
# 셀 2. 서버 실수율
MODEL_PATH = "yolo11n.pt" # 상수는 대문자로
CONF_THRESHOLD = 0.3

# 클래스명 -> 한글 + 기본 심각도
CLASS_MAP = {
    "bus":   ("버스", 2),
    "person": ("사람",1),
    "car":    ("자동차", 2),
    "truck":  ("트럭", 3),
}

_model = None # 모델은 전역에 한 번만 로드
def _get_model():
  global _model # 전역변수
  if _model is None:
    print(" (모델 로딩 - 서버 시작 후 첫 요청에만 발생)")
    _model = YOLO(MODEL_PATH) # 한 번만 로드해서 전역에 저장
  return _model

In [13]:
# 셀 3. _real_detect
# 이미지에서 위험 요소를 감지해 출력 예시 형태의 리스트로 반환
def _real_detect(img_path: str) -> list:          # def 함수(인자: 타입) -> 반환타입:
  model = _get_model()
  result = model(img_path, conf=CONF_THRESHOLD, verbose=False)[0] # CONF_THRESHOLD 이상의 값만 감지
  detections = []
  for box in result.boxes:  # 감지된 박스 하나씩
    name = result.names[int(box.cls)]
    if name not in CLASS_MAP:
      continue
    kor_name, severity = CLASS_MAP[name]
    x1, y1, x2, y2 = box.xyxyn[0].tolist() # 비율 좌표 텐서
    detections.append({
        "type": kor_name,
        "confidence": round(float(box.conf), 3),
        "severity": severity,
        "box": {"x1":round(x1, 3),"y1":round(y1,3),"x2":round(x2,3),"y2":round(y2, 3)},
    })
  return detections # 완성된 리스트 반환

In [14]:
# 셀 4. 실행 + 검증
detections = _real_detect("bus.jpg")
print(f"\n감지 {len(detections)}건:")
print(json.dumps(detections, ensure_ascii=False, indent=2)) # JSON문자열로 변환

# 검증
for d in detections:
  assert isinstance(d["type"], str) # instance (값, 타입): 타입이 맞나 확인, # assert 조건: 거짓이면 에러로 멈춤 참이면 둠
  assert isinstance(d["confidence"], float) and 0 <= d["confidence"] <= 1
  assert isinstance(d["severity"], int) and 1 <= d["severity"] <= 5
  assert set(d["box"].keys()) == {"x1", "y1", "x2", "y2"} #  키 구성이 4개인지 비교
  for v in d["box"].values():
    assert isinstance(v, float) and 0 <= v <= 1, "비율 좌표 위반!" # , 실패 이미지
print("출력 검증 통과 - mock과 같은 형식")

_ = _real_detect("bus.jpg") # _ = 반환값 안 쓰고 버림
print("두 번째 호출은 모델 없이 즉시 함수 실행")

 (모델 로딩 - 서버 시작 후 첫 요청에만 발생)

감지 5건:
[
  {
    "type": "버스",
    "confidence": 0.94,
    "severity": 2,
    "box": {
      "x1": 0.005,
      "y1": 0.212,
      "x2": 0.983,
      "y2": 0.674
    }
  },
  {
    "type": "사람",
    "confidence": 0.888,
    "severity": 1,
    "box": {
      "x1": 0.828,
      "y1": 0.366,
      "x2": 1.0,
      "y2": 0.814
    }
  },
  {
    "type": "사람",
    "confidence": 0.878,
    "severity": 1,
    "box": {
      "x1": 0.059,
      "y1": 0.37,
      "x2": 0.295,
      "y2": 0.837
    }
  },
  {
    "type": "사람",
    "confidence": 0.856,
    "severity": 1,
    "box": {
      "x1": 0.275,
      "y1": 0.378,
      "x2": 0.425,
      "y2": 0.797
    }
  },
  {
    "type": "사람",
    "confidence": 0.622,
    "severity": 1,
    "box": {
      "x1": 0.0,
      "y1": 0.515,
      "x2": 0.085,
      "y2": 0.808
    }
  }
]
출력 검증 통과 - mock과 같은 형식
두 번째 호출은 모델 없이 즉시 함수 실행


In [11]:
print("CLASS_MAP =", CLASS_MAP)
print("detections =", detections)

for d in detections:
    print(d["type"], type(d["type"]))


CLASS_MAP = {'bus': {2, '버스'}, 'person': {1, '사람'}, 'car': {2, '자동차'}, 'truck': {'트럭', 3}}
detections = [{'type': 2, 'confidence': 0.94, 'severity': '버스', 'box': {'x1': 0.005, 'y1': 0.212, 'x2': 0.983, 'y2': 0.674}}, {'type': 1, 'confidence': 0.888, 'severity': '사람', 'box': {'x1': 0.828, 'y1': 0.366, 'x2': 1.0, 'y2': 0.814}}, {'type': 1, 'confidence': 0.878, 'severity': '사람', 'box': {'x1': 0.059, 'y1': 0.37, 'x2': 0.295, 'y2': 0.837}}, {'type': 1, 'confidence': 0.856, 'severity': '사람', 'box': {'x1': 0.275, 'y1': 0.378, 'x2': 0.425, 'y2': 0.797}}, {'type': 1, 'confidence': 0.622, 'severity': '사람', 'box': {'x1': 0.0, 'y1': 0.515, 'x2': 0.085, 'y2': 0.808}}]
2 <class 'int'>
1 <class 'int'>
1 <class 'int'>
1 <class 'int'>
1 <class 'int'>
